# Session 05 — Object Detection and Shape Analysis

**Goal:** use contours and simple geometry to count, classify, and measure simple shapes. This is a rule-based method: it works when the objects are clear and separate.


## 1. Get the drawing tools

OpenCV finds boundaries. NumPy stores the image as numbers. Matplotlib shows the result.


In [ ]:
# Colab already includes OpenCV, NumPy, and Matplotlib.
# Import the image tools we will use.
import cv2
import numpy as np
import matplotlib.pyplot as plt


## 2. Make a controlled test image

We start with a clear circle, rectangle, and triangle. This lets us see exactly what a contour rule can do.


In [ ]:
# Make a blank white image: 420 pixels high and 640 pixels wide.
image = np.full((420, 640, 3), 255, dtype=np.uint8)

# Draw three filled black shapes.
cv2.circle(image, (120, 210), 65, (0, 0, 0), -1)
cv2.rectangle(image, (260, 145), (400, 275), (0, 0, 0), -1)
triangle = np.array([[500, 285], [565, 145], [620, 285]])
cv2.fillPoly(image, [triangle], (0, 0, 0))

# Turn colour into one brightness value for each pixel.
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# White shapes in this next image are the parts we want to find.
THRESHOLD = 127  # Try changing this value later.
_, binary = cv2.threshold(gray, THRESHOLD, 255, cv2.THRESH_BINARY_INV)

# Show the input and the black-and-white mask.
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original drawing")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(binary, cmap="gray")
plt.title("Binary mask")
plt.axis("off")
plt.show()


## 3. Find, label, and measure each contour

A contour is a list of points around one connected boundary. We use the number of simplified corners as a small rule for the shape name.


In [ ]:
# Find the outside boundary of every white connected shape.
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Copy the image so the original stays unchanged.
result = image.copy()

# Check one contour at a time.
for contour in contours:
    # Area is the number of pixels inside the boundary.
    area = cv2.contourArea(contour)

    # Perimeter is the distance around the boundary.
    perimeter = cv2.arcLength(contour, True)

    # Simplify the boundary into a few corner points.
    simple_shape = cv2.approxPolyDP(contour, 0.04 * perimeter, True)
    corners = len(simple_shape)

    # Make one easy-to-explain rule from the number of corners.
    if corners == 3:
        name = "triangle"
    elif corners == 4:
        name = "rectangle"
    else:
        name = "circle"

    # Draw a box that shows where this object was found.
    x, y, width, height = cv2.boundingRect(contour)
    cv2.rectangle(result, (x, y), (x + width, y + height), (21, 138, 194), 2)

    # Write the rule result and its measurement on the image.
    label = f"{name}: {area:.0f} px"
    cv2.putText(result, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (246, 138, 69), 2)

# Report the total before we display it.
print("Objects counted:", len(contours))

# Display the marked result.
plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


## 4. Stress-test the rule

Change only `THRESHOLD`, rerun the notebook, and write down what changes. Then answer: would this rule still work if objects overlapped, were rotated, or had a busy background?

**Evidence record:**
- Shape rule used: ______
- Objects counted: ______
- First condition that breaks it: ______
- When a deep detector would be more useful: ______
